<a href="https://colab.research.google.com/github/lakshmikanth448/CODSOFT/blob/main/customer_churn_telecom_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [95]:
import pandas as pd
df=pd.read_csv("/content/sample_data/customer_churn_telecom (4).csv");

In [96]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [97]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
7038,False
7039,False
7040,False
7041,False


In [98]:
df.isna().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [141]:
X=df.drop(['Churn', 'customerID'], axis=1)
y=df['Churn']

In [115]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(y)

print(y)

[0 0 1 ... 0 1 0]


In [116]:
num_cols=X.select_dtypes(include='number').columns
cat_cols=X.select_dtypes(include='object').columns

In [119]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler


In [120]:
preprocessor=ColumnTransformer(transformers=[
    ("Enocding",OneHotEncoder(drop="first"),cat_cols),
    ("Scaling",StandardScaler(),num_cols)
])

In [121]:
X_processed = preprocessor.fit_transform(X)

print(X_processed.shape)

(7043, 13601)


In [129]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [130]:
model=Sequential()
model.add(Dense(128,activation='relu',input_shape=(X_processed.shape[1],)))
model.add(Dropout(0.3))
model.add(Dense(64,activation='relu'))
model.add(Dense(32,activation='relu'))
model.add(Dense(16,activation='relu'))
model.add(Dense(8,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [132]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [133]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [134]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=4,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 28s 22ms/step - accuracy: 0.7823 - loss: 0.4568 - val_accuracy: 0.8083 - val_loss: 0.4126
Epoch 2/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 26s 23ms/step - accuracy: 0.8489 - loss: 0.3468 - val_accuracy: 0.7835 - val_loss: 0.4453
Epoch 3/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - accuracy: 0.9700 - loss: 0.0844 - val_accuracy: 0.7374 - val_loss: 0.6374
Epoch 4/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.9942 - loss: 0.0193 - val_accuracy: 0.7507 - val_loss: 0.6068
Epoch 5/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 24s 21ms/step - accuracy: 0.9960 - loss: 0.0092 - val_accuracy: 0.7551 - val_loss: 0.6847
Epoch 6/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 23s 20ms/step - accuracy: 0.9996 - loss: 0.0027 - val_accuracy: 0.7728 - val_loss: 0.8377
Epoch 7/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - accuracy: 1.0000 - loss: 8.9742e-05 - val_accuracy: 0.7453 - val_loss: 0.9930
Epoch 8/100
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 23s 20ms/step - accura

In [136]:
loss, accuracy = model.evaluate(X_processed, y)

print("Accuracy:", accuracy)

221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8212 - loss: 0.3914
Accuracy: 0.8212409615516663


In [137]:
pred = model.predict(X_test)

print(pred)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
[[0.67811614]
 [0.10189838]
 [0.01311133]
 ...
 [0.06305482]
 [0.02099594]
 [0.35156742]]


In [138]:
import numpy as np

pred_class = np.argmax(pred, axis=1)

print(pred_class)

[0 0 0 ... 0 0 0]


In [139]:
print(le.inverse_transform(pred_class))

[0 0 0 ... 0 0 0]


In [140]:
model.save("customer_churn_telecom.h5")

In [142]:
!pip install streamlit joblib
import joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.8 MB/s eta 0:00:00


In [143]:
# Save the preprocessor and label encoder
joblib.dump(preprocessor, 'preprocessor.joblib')
joblib.dump(le, 'label_encoder.joblib')
print('Preprocessor and LabelEncoder saved successfully.')

Preprocessor and LabelEncoder saved successfully.


In [144]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import joblib

# Load the trained model, preprocessor, and label encoder
model = load_model('customer_churn_telecom.h5')
preprocessor = joblib.load('preprocessor.joblib')
label_encoder = joblib.load('label_encoder.joblib')

st.title('Customer Churn Prediction')
st.write('Enter customer details to predict churn.')

# Input fields for user data
# Based on the columns in X=df.drop(['Churn', 'customerID'], axis=1)
# and considering cat_cols and num_cols from earlier processing

gender = st.selectbox('Gender', ['Male', 'Female'])
SeniorCitizen = st.selectbox('Senior Citizen', [0, 1])
Partner = st.selectbox('Partner', ['Yes', 'No'])
Dependents = st.selectbox('Dependents', ['Yes', 'No'])
tenure = st.slider('Tenure (months)', 0, 72, 1)
PhoneService = st.selectbox('Phone Service', ['Yes', 'No'])
MultipleLines = st.selectbox('Multiple Lines', ['No phone service', 'No', 'Yes'])
InternetService = st.selectbox('Internet Service', ['DSL', 'Fiber optic', 'No'])
OnlineSecurity = st.selectbox('Online Security', ['No', 'Yes', 'No internet service'])
OnlineBackup = st.selectbox('Online Backup', ['No', 'Yes', 'No internet service'])
DeviceProtection = st.selectbox('Device Protection', ['No', 'Yes', 'No internet service'])
TechSupport = st.selectbox('Tech Support', ['No', 'Yes', 'No internet service'])
StreamingTV = st.selectbox('Streaming TV', ['No', 'Yes', 'No internet service'])
StreamingMovies = st.selectbox('Streaming Movies', ['No', 'Yes', 'No internet service'])
Contract = st.selectbox('Contract', ['Month-to-month', 'One year', 'Two year'])
PaperlessBilling = st.selectbox('Paperless Billing', ['Yes', 'No'])
PaymentMethod = st.selectbox('Payment Method', ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'])
MonthlyCharges = st.number_input('Monthly Charges', min_value=0.0, max_value=120.0, value=50.0)
TotalCharges = st.number_input('Total Charges', min_value=0.0, max_value=9000.0, value=500.0)

# Create a DataFrame from user input
input_data = pd.DataFrame({
    'gender': [gender],
    'SeniorCitizen': [SeniorCitizen],
    'Partner': [Partner],
    'Dependents': [Dependents],
    'tenure': [tenure],
    'PhoneService': [PhoneService],
    'MultipleLines': [MultipleLines],
    'InternetService': [InternetService],
    'OnlineSecurity': [OnlineSecurity],
    'OnlineBackup': [OnlineBackup],
    'DeviceProtection': [DeviceProtection],
    'TechSupport': [TechSupport],
    'StreamingTV': [StreamingTV],
    'StreamingMovies': [StreamingMovies],
    'Contract': [Contract],
    'PaperlessBilling': [PaperlessBilling],
    'PaymentMethod': [PaymentMethod],
    'MonthlyCharges': [MonthlyCharges],
    'TotalCharges': [TotalCharges]
})

if st.button('Predict Churn'):
    # Preprocess the input data
    processed_input = preprocessor.transform(input_data)

    # Make prediction
    prediction_proba = model.predict(processed_input)[0][0]
    prediction_class = (prediction_proba > 0.5).astype(int)

    # Inverse transform to get original label (Yes/No)
    churn_status = label_encoder.inverse_transform([prediction_class])[0]

    st.subheader('Prediction Results')
    if churn_status == 'Yes':
        st.error(f'The customer is likely to CHURN (Probability: {prediction_proba:.2f})')
    else:
        st.success(f'The customer is likely NOT to CHURN (Probability: {prediction_proba:.2f})')


Writing app.py


### To run the Streamlit app:
1.  **Run the cells above** to install libraries, save the model components, and create `app.py`.
2.  **Open a new terminal** in Colab (File > New > New notebook, then Environment > Open new shell).
3.  **Navigate to the directory** where `app.py` is saved (usually `/content/`).
4.  **Run the command:** `streamlit run app.py & npx localtunnel --port 8501`
5.  **Click the external URL** provided by `localtunnel` to access your app in a new browser tab.

*(Note: If `npx` is not found, install node.js with `!npm install -g npx` first)*